# NICMOS Binary Search — Master Pipeline

This notebook runs the cleaned pipeline **filter-by-filter**:

**MAST dataset → target → filter → Grid → BFGS → HMC → diagnostic plot**

### Before running
- Put this notebook in the same `notebooks/` folder as your cleaned Grid, BFGS, and HMC notebooks.
- The three stage notebooks should now mainly contain reusable functions and should **not** contain bottom cells that automatically launch a manual fit.
- In the settings cell below, change the three notebook filenames if yours are named differently.

For the first test, this notebook is set to run **F170M only** with short `20/20` HMC.

## 1. Imports and project paths

In [36]:
from pathlib import Path
from collections import defaultdict
import pickle
import sys

import jax
import jax.numpy as np
from astropy.io import fits

jax.config.update("jax_enable_x64", True)

# ------------------------------------------------------------
# Find the project root whether this notebook is launched from
# notebooks/ or from the repository root.
# ------------------------------------------------------------
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    NOTEBOOK_DIR = cwd
    PROJECT_ROOT = cwd.parent
elif (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
    NOTEBOOK_DIR = cwd / "notebooks"
else:
    # Fallback: assume current directory is the notebooks folder
    NOTEBOOK_DIR = cwd
    PROJECT_ROOT = cwd.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models import *
from detectors import *
from apertures import *

print("PROJECT_ROOT :", PROJECT_ROOT)
print("NOTEBOOK_DIR :", NOTEBOOK_DIR)


PROJECT_ROOT : /Users/haarshakrishna/hubble
NOTEBOOK_DIR : /Users/haarshakrishna/hubble/FOSE7902


## 2. Pipeline settings

In [37]:
# ============================================================
# DATASET
# ============================================================

DATASET = "MAST_10143"

DATA_DIR = (
    NOTEBOOK_DIR
    / "data"
    / DATASET
    / "MAST"
    / "HST"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / DATASET
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)



GRID_NOTEBOOK = "Grid Search.ipynb"
BFGS_NOTEBOOK = "BFGS optimisation.ipynb"
HMC_NOTEBOOK = "HMC sampling.ipynb"


# ============================================================
# WHAT TO RUN
# ============================================================

# First smoke test:
FILTERS_TO_RUN = ["F170M"]

# None = every target found in the dataset
TARGETS_TO_RUN = None

# False for the very first test so every stage definitely runs.
# After a successful test, change this to True.
RESUME = False


# ============================================================
# PLOTTING
# ============================================================

MAKE_GRID_PLOTS = False
MAKE_BFGS_PLOTS = False
MAKE_HMC_PLOT = True
SHOW_HMC_PLOT = False


print("DATA_DIR    :", DATA_DIR)
print("RESULTS_DIR :", RESULTS_DIR)


DATA_DIR    : /Users/haarshakrishna/hubble/FOSE7902/data/MAST_10143/MAST/HST
RESULTS_DIR : /Users/haarshakrishna/hubble/results/MAST_10143


In [38]:
from pathlib import Path
import nbformat

print("Notebook directory:")
print(NOTEBOOK_DIR)

print("\nNotebooks found:\n")

for path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):

    print(
        path.name,
        "|",
        path.stat().st_size,
        "bytes"
    )

    try:
        nbformat.read(
            path,
            as_version=4
        )

        print("   ✓ valid notebook")

    except Exception as e:

        print(
            "   ✗ INVALID:",
            type(e).__name__,
            e
        )

Notebook directory:
/Users/haarshakrishna/hubble/FOSE7902

Notebooks found:

00_inspect_filters.ipynb | 37841 bytes
   ✓ valid notebook
BFGS optimisation.ipynb | 616438 bytes
   ✓ valid notebook
Grid Search.ipynb | 37360 bytes
   ✓ valid notebook
HMC sampling.ipynb | 51364 bytes
   ✓ valid notebook
better-grid-search-haarsha.ipynb | 514598 bytes
   ✓ valid notebook
binary_grid_search_debug.ipynb | 1294959 bytes
   ✓ valid notebook
calibrator-fit-new copy.ipynb | 1570032 bytes
   ✓ valid notebook
dlux.ipynb | 139471 bytes
   ✓ valid notebook
exploration.ipynb | 510162 bytes
   ✓ valid notebook
extra.ipynb | 18784 bytes
   ✓ valid notebook
load_data_ 10879.ipynb | 585952 bytes
   ✓ valid notebook
load_data_ 9737_wd.ipynb | 710496 bytes
   ✓ valid notebook
load_data_ 9833.ipynb | 764304 bytes
   ✓ valid notebook
load_data_10143.ipynb | 628181 bytes
   ✓ valid notebook
model.ipynb | 2285042 bytes
   ✓ valid notebook
run_pipeline.ipynb | 22835 bytes
   ✓ valid notebook
single_source_setup.i

## 3. Load the three cleaned stage notebooks

This brings `run_grid_for_group()`, `run_bfgs_for_group()`, `run_hmc_for_group()` and their helper functions into this notebook.

If this cell says a notebook cannot be found, just correct the three filenames in the previous cell.

In [39]:
def load_stage_notebook(filename):
    path = NOTEBOOK_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find stage notebook:\n{path}\n\n"
            "Change the filename in the settings cell."
        )

    print("Loading:", path.name)
    get_ipython().run_line_magic("run", f'"{path}"')


load_stage_notebook(GRID_NOTEBOOK)
load_stage_notebook(BFGS_NOTEBOOK)
load_stage_notebook(HMC_NOTEBOOK)

required_functions = [
    "run_grid_for_group",
    "run_bfgs_for_group",
    "run_hmc_for_group",
]

for name in required_functions:
    if name not in globals():
        raise RuntimeError(
            f"{name} was not defined after loading the stage notebooks."
        )

print("\nAll three pipeline functions are available.")


Loading: Grid Search.ipynb
Loading: BFGS optimisation.ipynb
Loading: HMC sampling.ipynb

All three pipeline functions are available.


## 4. Central model, Grid, BFGS and HMC configuration

In [40]:
# ============================================================
# COMMON MODEL SETTINGS
# ============================================================

MODEL_CONFIG = {
    "wid": 64,
    "oversample": 4,
    "psf_oversample": 4,
    "nwavels": 20,
    "npoly": 5,
    "n_zernikes": 20,
}


# ============================================================
# GRID
# ============================================================

GRID_CONFIG = {
    **MODEL_CONFIG,

    "n_aberrations": 26,

    "x_min": -5,
    "x_max": 5,
    "y_min": -5,
    "y_max": 5,
    "Nx": 50,
    "Ny": 50,

    "r_min": 0,
    "r_max": 5.0,
    "Nr": 20,

    "theta_min": -180,
    "theta_max": 180,
    "Ntheta": 50,

    "c_min": 1.0,
    "c_max": 10.0,
    "Nc": 20,
}


# ============================================================
# BFGS
# ============================================================

BFGS_CONFIG = {
    "max_steps": 100,
    "n_zernikes": MODEL_CONFIG["n_zernikes"],
    "oversample": MODEL_CONFIG["oversample"],

    "single_bfgs_keys": [
        "positions",
        "spectrum",
        "aberrations",
        "cold_mask_shift",
        "bias",
        "despace",
        "mag",
    ],

    "binary_bfgs_keys": [
        "positions",
        "separation",
        "position_angle",
        "primary_spectrum",
        "secondary_spectrum",
        "aberrations",
        "cold_mask_shift",
        "bias",
        "despace",
        "mag",
    ],

    "single_scale_values": {
        "positions": 0.1,
        "spectrum": 50.0,
        "aberrations": 1.0,
        "cold_mask_shift": 0.1,
        "bias": 0.01,
        "despace": 1.0,
        "mag": 0.1,
    },

    "binary_scale_values": {
        "positions": 0.1,
        "separation": 0.1,
        "position_angle": 1.0,
        "primary_spectrum": 50.0,
        "secondary_spectrum": 50.0,
        "aberrations": 1.0,
        "cold_mask_shift": 0.1,
        "bias": 0.01,
        "despace": 1.0,
        "mag": 0.1,
    },
}


# ============================================================
# HMC — screening run
# ============================================================

HMC_CONFIG = {
    "num_warmup": 20,
    "num_samples": 20,
    "seed": 0,
}


print("Configuration ready.")


Configuration ready.


## 5. Build the shared NICMOS model objects

In [41]:
wid = MODEL_CONFIG["wid"]
oversample = MODEL_CONFIG["oversample"]
psf_oversample = MODEL_CONFIG["psf_oversample"]
nwavels = MODEL_CONFIG["nwavels"]
npoly = MODEL_CONFIG["npoly"]
n_zernikes = MODEL_CONFIG["n_zernikes"]

# Same spectrum basis used in the current notebooks
spectrum_basis = np.ones(
    (nwavels, npoly)
)

# ------------------------------------------------------------
# Grid optics
# ------------------------------------------------------------
grid_optics = NICMOSOptics(
    512,
    wid,
    oversample
)

grid_oversampled_optics = NICMOSOptics(
    512,
    wid,
    oversample * psf_oversample,
    psf_oversample=psf_oversample
)

# ------------------------------------------------------------
# Full optics used by BFGS and HMC
# ------------------------------------------------------------
science_optics = NICMOSSecondaryFresnelOptics(
    512,
    wid,
    oversample,
    mag=3.3,
    defocus=0.0,
    despace=0.0,
    n_zernikes=n_zernikes
)

detector = NICMOSDetector(
    oversample,
    wid
    
)

print("Model objects ready.")


Model objects ready.


## 6. Discover every calibrated FITS file and separate by target + filter

The important line here is:

```python
key = (target, filt)
```

so F110W, F145M, F170M, etc. are handled independently.

In [42]:
files = sorted(
    DATA_DIR.glob("*_cal.fits")
)

if len(files) == 0:
    raise FileNotFoundError(
        f"No *_cal.fits files found in:\n{DATA_DIR}"
    )

print(f"Found {len(files)} calibrated FITS file(s).\n")

groups = defaultdict(
    lambda: {
        "single": [],
        "binary": [],
        "files": [],
    }
)

for fname in files:

    with fits.open(fname) as hdul:
        target = str(
            hdul[0].header["TARGNAME"]
        ).strip()

        filt = str(
            hdul[0].header["FILTER"]
        ).strip()

    # Optional target filtering
    if (
        TARGETS_TO_RUN is not None
        and target not in TARGETS_TO_RUN
    ):
        continue

    # Optional filter filtering
    if (
        FILTERS_TO_RUN is not None
        and filt not in FILTERS_TO_RUN
    ):
        continue

    exp_single = exposure_from_file(
        str(fname),
        SinglePointFit(
            spectrum_basis,
            filt
        ),
        crop=wid
    )

    exp_binary = exposure_from_file(
        str(fname),
        BinaryFit(
            spectrum_basis,
            filt
        ),
        crop=wid
    )

    key = (
        target,
        filt
    )

    groups[key]["single"].append(
        exp_single
    )

    groups[key]["binary"].append(
        exp_binary
    )

    groups[key]["files"].append(
        str(fname)
    )

    print(
        fname.name,
        "→",
        target,
        "→",
        filt
    )

if len(groups) == 0:
    raise RuntimeError(
        "No target/filter groups matched the current selection."
    )


Found 272 calibrated FITS file(s).

Version 4.4.1
crop centre: 253 106
Version 4.4.1
crop centre: 253 106
n8yj01rqq_cal.fits → U20373 → F170M
Version 4.4.1
crop centre: 69 158
Version 4.4.1
crop centre: 69 158
n8yj01rsq_cal.fits → U20373 → F170M
Version 4.4.1
crop centre: 33 41
Version 4.4.1
crop centre: 33 41
n8yj02wxq_cal.fits → U11296 → F170M
Version 4.4.1
crop centre: 79 41
Version 4.4.1
crop centre: 79 41
n8yj02wzq_cal.fits → U11296 → F170M
Version 4.4.1
crop centre: 210 67
Version 4.4.1
crop centre: 210 67
n8yj03tfq_cal.fits → U10742 → F170M
Version 4.4.1
crop centre: 255 67
Version 4.4.1
crop centre: 255 67
n8yj03thq_cal.fits → U10742 → F170M
Version 4.4.1
crop centre: 69 158
Version 4.4.1
crop centre: 69 158
n8yj04alq_cal.fits → U11314 → F170M
Version 4.4.1
crop centre: 69 158
Version 4.4.1
crop centre: 69 158
n8yj04anq_cal.fits → U11314 → F170M
Version 4.4.1
crop centre: 111 113
Version 4.4.1
crop centre: 111 113
n8yj05knq_cal.fits → U20976 → F170M
Version 4.4.1
crop centre: 1

## 7. Check the groups before doing any expensive fitting

In [43]:
print("=" * 70)
print("GROUPS TO RUN")
print("=" * 70)

for (target, filt), group in sorted(groups.items()):

    print(
        f"{target:30s} | "
        f"{filt:8s} | "
        f"{len(group['single'])} exposure(s)"
    )


GROUPS TO RUN
U10088                         | F170M    | 2 exposure(s)
U10102                         | F170M    | 2 exposure(s)
U10151                         | F170M    | 2 exposure(s)
U10287                         | F170M    | 2 exposure(s)
U10312                         | F170M    | 2 exposure(s)
U10329                         | F170M    | 2 exposure(s)
U10390                         | F170M    | 2 exposure(s)
U10601                         | F170M    | 2 exposure(s)
U10617                         | F170M    | 2 exposure(s)
U10721                         | F170M    | 2 exposure(s)
U10742                         | F170M    | 2 exposure(s)
U10764                         | F170M    | 2 exposure(s)
U10802                         | F170M    | 2 exposure(s)
U10929                         | F170M    | 2 exposure(s)
U10954                         | F170M    | 2 exposure(s)
U10960                         | F170M    | 2 exposure(s)
U11044                         | F170M    | 2 exposure(s)


## 8. Run Grid → BFGS → HMC

For the first test, leave `FILTERS_TO_RUN = ["F170M"]` and `RESUME = False`.

After this works, change:
```python
FILTERS_TO_RUN = None
RESUME = True
```
to run/continue the whole dataset filter-by-filter.

In [44]:
pipeline_results = {}


def safe_name(value):
    return (
        str(value)
        .replace(" ", "_")
        .replace("/", "_")
    )


for (target, filt), group in sorted(groups.items()):

    print()
    print("#" * 70)
    print("TARGET    :", target)
    print("FILTER    :", filt)
    print("EXPOSURES :", len(group["single"]))
    print("#" * 70)

    exposures_single = group["single"]
    exposures_binary = group["binary"]

    group_dir = (
        RESULTS_DIR
        / safe_name(target)
        / filt
    )

    group_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    grid_path = (
        group_dir
        / "grid.pkl"
    )

    bfgs_path = (
        group_dir
        / "bfgs.pkl"
    )

    hmc_path = (
        group_dir
        / "hmc.pkl"
    )


    # ========================================================
    # GRID
    # ========================================================

    if RESUME and grid_path.exists():

        print("\nLoading existing Grid result:")
        print(grid_path)

        with open(
            grid_path,
            "rb"
        ) as f:
            grid_output = pickle.load(f)

    else:

        grid_output = run_grid_for_group(
            target=target,
            filt=filt,
            exposures_single=exposures_single,
            exposures_binary=exposures_binary,
            optics=grid_optics,
            oversampled_optics=grid_oversampled_optics,
            detector=detector,
            spectrum_basis=spectrum_basis,
            config=GRID_CONFIG,
            results_dir=RESULTS_DIR,
            make_plots=MAKE_GRID_PLOTS,
            save=True,
        )


    # ========================================================
    # BFGS
    # ========================================================

    if RESUME and bfgs_path.exists():

        print("\nLoading existing BFGS result:")
        print(bfgs_path)

        with open(
            bfgs_path,
            "rb"
        ) as f:
            bfgs_output = pickle.load(f)

    else:

        bfgs_output = run_bfgs_for_group(
            target=target,
            filt=filt,
            exposures_single=exposures_single,
            exposures_binary=exposures_binary,
            grid_output=grid_output,
            optics=science_optics,
            detector=detector,
            config=BFGS_CONFIG,
            results_dir=RESULTS_DIR,
            make_plots=MAKE_BFGS_PLOTS,
            save=True,
        )


    # ========================================================
    # HMC
    # ========================================================

    if RESUME and hmc_path.exists():

        print("\nLoading existing HMC result:")
        print(hmc_path)

        with open(
            hmc_path,
            "rb"
        ) as f:
            hmc_output = pickle.load(f)

    else:

        (
            hmc_output,
            single_mcmc,
            binary_mcmc,
            model_single,
            model_binary,
        ) = run_hmc_for_group(
            target=target,
            filt=filt,
            exposures_single=exposures_single,
            exposures_binary=exposures_binary,
            bfgs_output=bfgs_output,
            optics=science_optics,
            detector=detector,
            config=HMC_CONFIG,
            results_dir=RESULTS_DIR,
            save=True,
            make_plot=MAKE_HMC_PLOT,
            show_plot=SHOW_HMC_PLOT,
        )


    pipeline_results[
        (target, filt)
    ] = {
        "grid": grid_output,
        "bfgs": bfgs_output,
        "hmc": hmc_output,
    }


print()
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

for target, filt in pipeline_results:
    print("✓", target, "|", filt)

print("\nResults directory:")
print(RESULTS_DIR)



######################################################################
TARGET    : U10088
FILTER    : F170M
EXPOSURES : 2
######################################################################

TARGET: U10088
FILTER: F170M


Single grid:   0%|          | 0/50 [00:00<?, ?it/s]

Best x shift:  0.7142857142857135
Best y shift:  0.9183673469387751
Single loss :  51371.35837634627
Single flux :  1829.1137516398958


Binary separation:   0%|          | 0/20 [00:00<?, ?it/s]

Best x shift: 0.7142857142857135
Best y shift: 0.9183673469387751
Best separation: 0.2631578947368421
Best position angle: 40.408163265306094
Best contrast: 10.0
Binary loss: 51142.87420932135
Binary flux: 1830.6704939153865
Saved: /Users/haarshakrishna/hubble/results/MAST_10143/U10088/F170M/grid.pkl

RUNNING BFGS
TARGET: U10088
FILTER: F170M
Grid results received
Single loss before: 7298874.836595746
Binary loss before: 7913962.963796386

Running single BFGS...

Running binary BFGS...

RESULTS
----------------------------------------
Single: 7298874.836595746 → 4071644.454076951
Binary: 7913962.963796386 → 2461.1894616658165
Single finite: True
Binary finite: True

Saved: /Users/haarshakrishna/hubble/results/MAST_10143/U10088/F170M/bfgs.pkl

RUNNING HMC
TARGET: U10088
FILTER: F170M
Loaded BFGS parameters

HMC settings
Warmup : 20
Samples: 20
Seed   : 0

Running SINGLE HMC...


sample: 100%|██████████| 40/40 [1:37:25<00:00, 146.13s/it, 5 steps of size 8.48e-01. acc. prob=0.68]     



Running BINARY HMC...


ValueError: Normal distribution got invalid loc parameter.

## 9. Inspect the saved output structure

Each target/filter should end up as:

```text
results/
└── MAST_10143/
    └── TARGET/
        └── F170M/
            ├── grid.pkl
            ├── bfgs.pkl
            ├── hmc.pkl
            └── plots/
                └── hmc_comparison.png
```


In [ ]:
for (target, filt) in pipeline_results:

    group_dir = (
        RESULTS_DIR
        / safe_name(target)
        / filt
    )

    print()
    print(target, "|", filt)

    if group_dir.exists():
        for path in sorted(group_dir.rglob("*")):
            if path.is_file():
                print(
                    "  ",
                    path.relative_to(RESULTS_DIR)
                )
